<a href="https://colab.research.google.com/github/GHROTH-L/-ai-ml-training-/blob/main/Predicting_Student_Test_Scores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pandas as pd
import numpy as np
import io
import matplotlib.pyplot as plt
import seaborn as sns #畫圖使用
%matplotlib inline
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.preprocessing import OneHotEncoder , LabelEncoder, TargetEncoder
from scipy.stats import f_oneway
from sklearn.model_selection import train_test_split , cross_val_score
from sklearn.metrics import accuracy_score, root_mean_squared_error
#將dataframe 下載下來
from google.colab import files

In [2]:

from google.colab import files
# 上傳
uploaded = files.upload()  #for train
uploaded2 = files.upload()  #for test

Saving train.csv to train.csv


Saving test.csv to test.csv


In [5]:
train = pd.read_csv('train.csv')   # 最一開始
test = pd.read_csv('test.csv')

In [6]:
train.describe()

,id,age,study_hours,class_attendance,sleep_hours,exam_score
count,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000
mean,314999.500000,20.545821,4.002337,71.987261,7.072758,62.506672
std,181865.479132,2.260238,2.359880,17.430098,1.744811,18.916884
min,0.000000,17.000000,0.080000,40.600000,4.100000,19.599000
25%,157499.750000,19.000000,1.970000,57.000000,5.600000,48.800000
50%,314999.500000,21.000000,4.000000,72.600000,7.100000,62.600000
75%,472499.250000,23.000000,6.050000,87.200000,8.600000,76.300000
max,629999.000000,24.000000,7.910000,99.400000,9.900000,100.000000


In [7]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                630000 non-null  int64  
 1   age               630000 non-null  int64  
 2   gender            630000 non-null  object 
 3   course            630000 non-null  object 
 4   study_hours       630000 non-null  float64
 5   class_attendance  630000 non-null  float64
 6   internet_access   630000 non-null  object 
 7   sleep_hours       630000 non-null  float64
 8   sleep_quality     630000 non-null  object 
 9   study_method      630000 non-null  object 
 10  facility_rating   630000 non-null  object 
 11  exam_difficulty   630000 non-null  object 
 12  exam_score        630000 non-null  float64
dtypes: float64(4), int64(2), object(7)
memory usage: 62.5+ MB


In [8]:
target_col='exam_score'

X=train.drop(columns=target_col)
y=train[target_col]

In [9]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [23]:
at_cols = ['gender', 'course', 'internet_access', 'sleep_quality', 'study_method',
            'facility_rating', 'exam_difficulty']

X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

base_features = ['study_hours', 'class_attendance', 'sleep_hours']
dummy_prefixes = ['sleep_quality_', 'study_method_', 'facility_rating_']

dummy_features = [
    col for col in X_train.columns
    if any(col.startswith(prefix) for prefix in dummy_prefixes)
]

features = base_features + dummy_features


In [31]:
X_train['study_efficiency'] = X_train['study_hours'] * X_train['class_attendance']
X_test['study_efficiency'] = X_test['study_hours'] * X_test['class_attendance']

X_train['study_sleep_ratio'] = X_train['study_hours'] / (X_train['sleep_hours'] + 1)
X_test['study_sleep_ratio'] = X_test['study_hours'] / (X_test['sleep_hours'] + 1)

In [36]:
print(X_train.head())

            id  age  study_hours  class_attendance  sleep_hours  gender_1  \
132470  132470   20         4.55              62.6          6.6     False   
230953  230953   18         4.83              91.1          5.9      True   
622969  622969   20         3.57              44.9          9.2      True   
77625    77625   22         7.91              85.4          6.4     False   
348317  348317   19         2.98              49.3          5.1     False   

        gender_2  course_1  course_2  course_3  ...  study_method_1  \
132470     False     False      True     False  ...           False   
230953     False     False      True     False  ...            True   
622969     False      True     False     False  ...           False   
77625       True     False      True     False  ...            True   
348317      True     False      True     False  ...           False   

        study_method_2  study_method_3  study_method_4  facility_rating_1  \
132470           False           

In [41]:


lr = LinearRegression()

rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42
)

gbr = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)


#lr.fit(X_train[features], y_train)
#rf.fit(X_train[features], y_train)
xgb.fit(X_train[features], y_train)
gbr.fit(X_train[features], y_train)


#pred_lr = lr.predict(X_test[features])
#pred_rf = rf.predict(X_test[features])
pred_xgb = xgb.predict(X_test[features])
pred_gbr = gbr.predict(X_test[features])

print(pred_xgb)
print(pred_rf)
print(pred_gbr)
y_pred = (
    0.8 * pred_xgb +
    0.2 * pred_gbr
)

rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)


[48.616768 56.682217 29.470634 ... 83.30971  54.067047 49.168232]
[48.53473163 54.9915286  31.73538033 ... 83.95630099 57.08712216
 47.20990765]
[49.61337666 57.08102215 30.48597025 ... 83.76271691 53.64244992
 50.3606071 ]
8.832695560589329


In [48]:
y_pred = (
    0.8 * pred_xgb +
    0.2 * pred_gbr
)

rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)

8.825098585862634


In [53]:
at_cols = ['gender', 'course', 'internet_access', 'sleep_quality', 'study_method',
            'facility_rating', 'exam_difficulty']

New_test = pd.get_dummies(test, columns=cat_cols, drop_first=True)


base_features = ['study_hours', 'class_attendance', 'sleep_hours']
dummy_prefixes = ['sleep_quality_', 'study_method_', 'facility_rating_']

dummy_features = [
    col for col in New_test.columns
    if any(col.startswith(prefix) for prefix in dummy_prefixes)
]


In [56]:
for col in cat_cols:
  test[col]=label_encoder.fit_transform(test[col])

y_test_pred_xgb = xgb.predict(New_test[features])
y_test_pred_gbr = gbr.predict(New_test[features])

y_test_pred= (
    0.8 * y_test_pred_xgb +
    0.2 * y_test_pred_gbr
)


In [57]:
submission = pd.DataFrame({
    'id': test.id,
    'exam_score': y_test_pred
})

submission.to_csv('submission.csv', index=False)

In [58]:
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>